## Step 1: Connect to GPU
Go to **Runtime** → **Change runtime type** and select **T4 GPU** (or any available GPU).

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Navigate to your project folder
PROJECT_PATH = '/content/drive/MyDrive/neuroseg'  # CHANGE THIS to your path
os.chdir(PROJECT_PATH)
print(f"Current Directory: {os.getcwd()}")

## Step 3: Install Dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless tqdm

## Step 6: Run Improved Training with Live Metrics

In [ ]:
!python3 train_improved.py

## Step 6b: Live Metrics Monitor (Optional - Run in Parallel)

Run this cell **simultaneously** with training above to see metrics update live every 5 seconds!

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
import time
import threading
import os

# Live monitoring function
def monitor_training_live():
    """Monitor training_history.json and plot metrics live"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('⏱️  LIVE Training Metrics - Maximum Dice Optimization', 
                 fontsize=14, fontweight='bold')
    
    history_file = 'training_history.json'
    
    while True:
        if os.path.exists(history_file):
            try:
                with open(history_file, 'r') as f:
                    history = json.load(f)
                
                # Plot 1: Loss
                axes[0, 0].clear()
                axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2, alpha=0.7)
                axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
                axes[0, 0].set_title('Loss', fontweight='bold')
                axes[0, 0].set_xlabel('Epoch')
                axes[0, 0].set_ylabel('Loss')
                axes[0, 0].legend()
                axes[0, 0].grid(True, alpha=0.3)
                
                # Plot 2: Dice Coefficient
                axes[0, 1].clear()
                axes[0, 1].plot(history['train_dice'], label='Train Dice', linewidth=2, alpha=0.7)
                axes[0, 1].plot(history['val_dice'], label='Val Dice', linewidth=2, color='orange')
                max_dice = max(history['val_dice'])
                axes[0, 1].axhline(y=0.90, color='g', linestyle='--', alpha=0.5, label='Target (0.90)')
                axes[0, 1].set_title(f'Dice Coefficient (Max: {max_dice:.4f})', fontweight='bold')
                axes[0, 1].set_xlabel('Epoch')
                axes[0, 1].set_ylabel('Dice')
                axes[0, 1].set_ylim([0, 1])
                axes[0, 1].legend()
                axes[0, 1].grid(True, alpha=0.3)
                
                # Plot 3: IoU
                axes[0, 2].clear()
                axes[0, 2].plot(history['train_iou'], label='Train IoU', linewidth=2, alpha=0.7)
                axes[0, 2].plot(history['val_iou'], label='Val IoU', linewidth=2, color='red')
                axes[0, 2].set_title('IoU Score', fontweight='bold')
                axes[0, 2].set_xlabel('Epoch')
                axes[0, 2].set_ylabel('IoU')
                axes[0, 2].set_ylim([0, 1])
                axes[0, 2].legend()
                axes[0, 2].grid(True, alpha=0.3)
                
                # Plot 4: Smoothed Dice (check for oscillations)
                axes[1, 0].clear()
                val_dice = history['val_dice']
                window = min(10, len(val_dice) // 5)
                if window > 1:
                    smoothed = np.convolve(val_dice, np.ones(window)/window, mode='valid')
                    axes[1, 0].plot(val_dice, label='Raw Dice', alpha=0.3, linewidth=1)
                    axes[1, 0].plot(range(window-1, len(val_dice)), smoothed, 
                                   label=f'Smoothed (window={window})', linewidth=2, color='orange')
                else:
                    axes[1, 0].plot(val_dice, label='Dice', linewidth=2)
                axes[1, 0].set_title('Smoothness Check (No Oscillations?)', fontweight='bold')
                axes[1, 0].set_xlabel('Epoch')
                axes[1, 0].set_ylabel('Dice')
                axes[1, 0].legend()
                axes[1, 0].grid(True, alpha=0.3)
                
                # Plot 5: Learning Rate
                axes[1, 1].clear()
                if 'learning_rates' in history and len(history['learning_rates']) > 0:
                    axes[1, 1].plot(history['learning_rates'], linewidth=2, color='purple')
                    axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
                    axes[1, 1].set_xlabel('Epoch')
                    axes[1, 1].set_ylabel('LR')
                    axes[1, 1].grid(True, alpha=0.3)
                else:
                    axes[1, 1].text(0.5, 0.5, 'LR tracking\nin progress...', 
                                   ha='center', va='center', fontsize=12)
                    axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
                
                # Plot 6: Generalization Gap
                axes[1, 2].clear()
                gap = [history['train_dice'][i] - history['val_dice'][i] 
                       for i in range(len(history['val_dice']))]
                axes[1, 2].plot(gap, linewidth=2, color='brown')
                axes[1, 2].axhline(y=0.05, color='g', linestyle='--', alpha=0.5, label='Good (0.05)')
                axes[1, 2].axhline(y=0.10, color='orange', linestyle='--', alpha=0.5, label='OK (0.10)')
                axes[1, 2].set_title('Train-Val Gap (Overfitting Check)', fontweight='bold')
                axes[1, 2].set_xlabel('Epoch')
                axes[1, 2].set_ylabel('Gap')
                axes[1, 2].legend()
                axes[1, 2].grid(True, alpha=0.3)
                
                plt.tight_layout()
                clear_output(wait=True)
                display(fig)
                
                # Show status
                epoch_num = len(history['val_dice'])
                current_dice = history['val_dice'][-1]
                max_dice = max(history['val_dice'])
                print(f"📊 Epoch {epoch_num:3d} | Val Dice: {current_dice:.4f} | Max: {max_dice:.4f}", end="")
                
                if current_dice >= 0.92:
                    print(" ✓✓ EXCELLENT!")
                elif current_dice >= 0.90:
                    print(" ✓ GREAT!")
                elif current_dice >= 0.85:
                    print(" ↗ Good progress")
                else:
                    print(" ...")
                
            except Exception as e:
                pass
        
        time.sleep(5)  # Update every 5 seconds

# Start live monitor in background
print("🟢 Starting LIVE metrics monitor...")
print("📊 Updates every 5 seconds during training\n")

monitor_thread = threading.Thread(target=monitor_training_live, daemon=True)
monitor_thread.start()

# Keep this cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n\n✓ Monitor stopped")

## Step 7: Load and Display Training Metrics

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load training history
with open('training_history.json', 'r') as f:
    history = json.load(f)

print("Training History Keys:", list(history.keys()))
print(f"\nTraining Epochs: {len(history['train_loss'])}")
print(f"Best Val Dice: {max(history['val_dice']):.4f}")
print(f"Final Val Dice: {history['val_dice'][-1]:.4f}")

## Step 8: Display Comprehensive Metrics Plot

In [ ]:
from IPython.display import Image
display(Image('training_metrics.png'))

## Step 9: Verify Results Met Requirements

Check if training met your strict requirements:

In [ ]:
import json

# Load results
with open('training_history.json', 'r') as f:
    history = json.load(f)

max_dice = max(history['val_dice'])
final_dice = history['val_dice'][-1]
train_val_gap = history['train_dice'][-1] - history['val_dice'][-1]
max_oscillation = max([abs(history['val_dice'][i] - history['val_dice'][i-1]) 
                       for i in range(40, len(history['val_dice']))])

print("="*60)
print("YOUR STRICT REQUIREMENTS - VERIFICATION")
print("="*60)

# Requirement 1: Maximum Dice
print(f"\n1. MAXIMUM DICE")
print(f"   Target: ≥ 0.90")
print(f"   Achieved: {max_dice:.4f}", end="")
if max_dice >= 0.92:
    print("  EXCELLENT (exceeds target!)")
elif max_dice >= 0.90:
    print("  MET (at target!)")
else:
    print(f"  MISSED (gap: {0.90 - max_dice:.4f})")

# Requirement 2: Smooth increasing curve
print(f"\n2. SMOOTH INCREASING CURVE")
changes = [abs(history['val_dice'][i] - history['val_dice'][i-1]) 
           for i in range(1, len(history['val_dice']))]
avg_change = np.mean(changes)
print(f"   Average per-epoch change: {avg_change:.4f}")
if avg_change < 0.005:
    print(f"   Status:  SMOOTH (excellent convergence)")
elif avg_change < 0.01:
    print(f"   Status:  ACCEPTABLE (good convergence)")
else:
    print(f"   Status:  ROUGH (needs improvement)")

# Requirement 3: No oscillations (after epoch 40)
print(f"\n3. NO OSCILLATIONS (after epoch 40)")
late_oscillation = max([abs(history['val_dice'][i] - history['val_dice'][i-1]) 
                        for i in range(min(40, len(history['val_dice'])), len(history['val_dice']))])
print(f"   Max change (epochs 40+): {late_oscillation:.4f}")
if late_oscillation < 0.02:
    print(f"   Status:  NO OSCILLATIONS")
elif late_oscillation < 0.05:
    print(f"   Status:  MINIMAL (acceptable)")
else:
    print(f"   Status:  OSCILLATING")

# Requirement 4: Generalization
print(f"\n4. GOOD GENERALIZATION")
print(f"   Train-Val Gap (final): {train_val_gap:.4f}")
if train_val_gap < 0.05:
    print(f"   Status:  EXCELLENT generalization")
elif train_val_gap < 0.10:
    print(f"   Status:  GOOD generalization")
else:
    print(f"   Status:  SLIGHT OVERFITTING")

print("\n" + "="*60)
print("FINAL VERDICT")
print("="*60)

all_good = (max_dice >= 0.90 and avg_change < 0.01 and 
            late_oscillation < 0.02 and train_val_gap < 0.10)

if all_good:
    print(" ALL REQUIREMENTS MET!")
    print(f"    Maximum Dice: {max_dice:.4f}")
    print(f"    Smooth Curves: {avg_change:.4f}")
    print(f"    No Oscillations: {late_oscillation:.4f}")
    print(f"    Good Generalization: {train_val_gap:.4f}")
    print("\n   Your NeuroSeg model is ready for deployment!")
else:
    print(" Some requirements not fully met, but training completed.")
    print(f"   Max Dice: {max_dice:.4f}")
    print("   Review metrics plot above for details.")

## Step 9: Analyze Training Stability

In [ ]:
import pandas as pd

# Calculate metrics stability
val_dice = np.array(history['val_dice'])
val_loss = np.array(history['val_loss'])

# Compute differences between consecutive epochs
dice_diffs = np.abs(np.diff(val_dice))
loss_diffs = np.abs(np.diff(val_loss))

print("Training Stability Analysis")
print("="*50)
print(f"\nVal Dice Stats:")
print(f"  Mean: {np.mean(val_dice):.4f}")
print(f"  Std Dev: {np.std(val_dice):.4f}")
print(f"  Min: {np.min(val_dice):.4f}")
print(f"  Max: {np.max(val_dice):.4f}")
print(f"  Mean Epoch-to-Epoch Change: {np.mean(dice_diffs):.4f}")
print(f"  Max Epoch-to-Epoch Change: {np.max(dice_diffs):.4f}")

print(f"\nVal Loss Stats:")
print(f"  Mean: {np.mean(val_loss):.4f}")
print(f"  Std Dev: {np.std(val_loss):.4f}")
print(f"  Min: {np.min(val_loss):.4f}")
print(f"  Max: {np.max(val_loss):.4f}")
print(f"  Mean Epoch-to-Epoch Change: {np.mean(loss_diffs):.4f}")
print(f"  Max Epoch-to-Epoch Change: {np.max(loss_diffs):.4f}")

# Detect oscillations
print(f"\nOscillation Detection:")
if np.max(dice_diffs) > 0.1:
    print(f"   Large jumps detected: {np.max(dice_diffs):.4f}")
    oscillating_epochs = np.where(dice_diffs > 0.05)[0]
    print(f"  Epochs with >0.05 changes: {oscillating_epochs}")
else:
    print(f"   Training is stable! Max change: {np.max(dice_diffs):.4f}")

## Step 10: Plot Training vs Validation Gap (Overfitting Indicator)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_dice']) + 1)

# Plot 1: Dice Comparison
axes[0].plot(epochs, history['train_dice'], 'o-', label='Train Dice', linewidth=2, markersize=4)
axes[0].plot(epochs, history['val_dice'], 's-', label='Val Dice', linewidth=2, markersize=4)
axes[0].fill_between(epochs, history['train_dice'], history['val_dice'], alpha=0.2)
axes[0].set_title('Training vs Validation Dice', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Dice Coefficient')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Plot 2: Gap (Overfitting indicator)
dice_gap = np.array(history['train_dice']) - np.array(history['val_dice'])
axes[1].plot(epochs, dice_gap, 'r^-', linewidth=2, markersize=6)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].fill_between(epochs, 0, dice_gap, alpha=0.3, color='red')
axes[1].set_title('Overfitting Indicator (Train - Val)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Gap')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_gap_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Average Training-Validation Gap: {np.mean(dice_gap):.4f}")
if np.mean(dice_gap) > 0.1:
    print(" Model may be overfitting - try more regularization")
else:
    print(" Good generalization - Train and Val performance align")

## Step 12: Save Model and Artifacts

In [ ]:
import shutil

# Copy best model and metrics to Google Drive
shutil.copy('best_model.pth', os.path.join(PROJECT_PATH, 'best_model.pth'))
shutil.copy('training_history.json', os.path.join(PROJECT_PATH, 'training_history.json'))
shutil.copy('training_metrics.png', os.path.join(PROJECT_PATH, 'training_metrics.png'))
shutil.copy('training_gap_analysis.png', os.path.join(PROJECT_PATH, 'training_gap_analysis.png'))

print("✓ All artifacts saved to Google Drive!")
print(f"  - best_model.pth")
print(f"  - training_history.json")
print(f"  - training_metrics.png")
print(f"  - training_gap_analysis.png")

## Summary

### Why First Epoch is Slow:
- Data loading, GPU warmup, kernel compilation
- **This is normal** - subsequent epochs are ~1.5 mins

### Why Training Was Oscillating:
- Small batch size (4) → noisy gradients
- Learning rate scheduler jumping around
- Missing gradient clipping
- Loss function misalignment

### What We Fixed:
- ✓ Batch size: 4 → 8
- ✓ Gradient clipping: Added
- ✓ Learning rate: Optimized
- ✓ Scheduler: Fixed with proper warmup
- ✓ Live metrics: Added visualization

### Expected Results:
- **Smooth training** without oscillations
- **0.87-0.92 Dice score** (matching Kaggle reference)
- **Stable convergence** after ~100 epochs
- **No sudden drops** in validation metrics